# 10:32 恢复动作的首小时半导体同业广度来源

固定 AMD/NVDA/SMH 09:35–10:30 完成 bar 的同涨比例。只做 Qlib 来源和时钟预检，0 标签、0 拟合、0 账户。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_first_hour_peer_breadth_action_source_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
s=plan['source'];assert sha(root/s['provider_uri']/'qlib_store_manifest.json')==s['provider_manifest_sha256'] and sha(root/s['parent_card']/'summary.json')==s['parent_summary_sha256'] and sha(root/s['reference_action_source'])==s['reference_action_summary_sha256']
parent=json.loads((root/s['parent_card']/'summary.json').read_text());assert parent['recorder_id']==s['parent_recorder_id'] and parent['nas_evidence']['sha256']==s['parent_NAS_archive_sha256']
readback=json.loads((root/s['parent_NAS_readback']).read_text());assert readback['NAS_archive_sha256']==s['parent_NAS_archive_sha256']
reference=json.loads((root/s['reference_action_source']).read_text());assert reference['source_NAS_archive_sha256']==s['parent_NAS_archive_sha256'] and reference['source_gate_passed']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/s['provider_uri']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2018-10-15 09:35',end_time='2026-10-08 10:30',freq='5min'));days=pd.DatetimeIndex(calendar.normalize().unique())
cols=['$open','$close','$price_valid'];raw=D.features(plan['feature']['instruments'],cols,start_time='2018-10-15 09:35',end_time='2026-10-08 10:30',freq='5min')
assert isinstance(raw.index,pd.MultiIndex) and set(raw.index.names)=={'instrument','datetime'} and raw.index.is_unique
bars=raw.reset_index();bars['datetime']=pd.to_datetime(bars['datetime']);bars['trade_date']=bars.datetime.dt.normalize();bars['clock']=bars.datetime.dt.strftime('%H:%M');clocks=plan['feature']['required_right_labelled_bars'];early=bars.loc[bars.clock.isin(clocks)].copy()
groups={(d,inst):g.sort_values('datetime') for (d,inst),g in early.groupby(['trade_date','instrument'],sort=False)}
rows=[];details=[]
for day in days:
 positive=[];status='known';invalid_reason=None
 for inst in plan['feature']['instruments']:
  g=groups.get((day,inst),early.iloc[0:0]);valid=len(g)==12 and g.clock.tolist()==clocks and g['$open'].notna().all() and g['$close'].notna().all() and np.isfinite(g[['$open','$close','$price_valid']].to_numpy(float)).all() and (g[['$open','$close']].to_numpy(float)>0).all() and g['$price_valid'].eq(1).all()
  if not valid:
   status='unknown';invalid_reason='missing_or_invalid_'+inst;details.append({'trade_date':day,'instrument':inst,'bars':len(g),'valid':False,'log_first_hour_return':np.nan});continue
  ret=float(np.log(float(g.iloc[-1]['$close'])/float(g.iloc[0]['$open'])));positive.append(ret>0);details.append({'trade_date':day,'instrument':inst,'bars':12,'valid':True,'log_first_hour_return':ret})
 fraction=float(np.mean(positive)) if status=='known' and len(positive)==3 else np.nan
 rows.append({'trade_date':day,'peer_positive_fraction_3':fraction,'source_known':bool(np.isfinite(fraction)),'status':status,'invalid_reason':invalid_reason,'assumed_available_at':day+pd.Timedelta(hours=10,minutes=31),'receiver_decision_at':day+pd.Timedelta(hours=10,minutes=32)})
daily=pd.DataFrame(rows);detail=pd.DataFrame(details);assert len(daily)==len(days) and daily.trade_date.equals(pd.Series(days,name='trade_date')) and detail.groupby('trade_date').size().eq(3).all()
train=daily.loc[daily.trade_date.between(*[pd.Timestamp(x) for x in plan['calendar']['train_action']])];held=daily.loc[daily.trade_date.between(*[pd.Timestamp(x) for x in plan['calendar']['reused_held_action']])]
audit={'status':'peer_breadth_source_gate_passed' if len(train)==plan['calendar']['expected_train_days'] and len(held)==plan['calendar']['expected_held_days'] and train.source_known.mean()>=plan['calendar']['minimum_known_fraction_each'] and held.source_known.mean()>=plan['calendar']['minimum_known_fraction_each'] else 'peer_breadth_source_gate_failed','plan_sha256':sha(card/'plan.json'),'source_calendar_days':len(days),'training_calendar_days':len(train),'training_known_days':int(train.source_known.sum()),'held_calendar_days':len(held),'held_known_days':int(held.source_known.sum()),'training_known_fraction':float(train.source_known.mean()),'held_known_fraction':float(held.source_known.mean()),'unknown_train_dates':[str(x.date()) for x in train.loc[~train.source_known,'trade_date']],'unknown_held_dates':[str(x.date()) for x in held.loc[~held.source_known,'trade_date']],'feature_value_counts':{str(k):int(v) for k,v in daily.peer_positive_fraction_3.value_counts(dropna=False).items()},'first_receipt_finality_qualified':False,'market_outcome_label_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n');assert audit['status']=='peer_breadth_source_gate_passed'
assert (daily.assumed_available_at<daily.receiver_decision_at).all()
static=pd.concat({'feature':daily.set_index('trade_date')[['peer_positive_fraction_3']].rename(columns={'peer_positive_fraction_3':plan['feature']['name']}),'meta':daily.set_index('trade_date')[['source_known','assumed_available_at','receiver_decision_at']]},axis=1);static.index=pd.MultiIndex.from_arrays([daily.trade_date,['SOXL_SOXX_CASH']*len(daily)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['calendar']['train_action']),'held':tuple(plan['calendar']['reused_held_action'])});tr=dataset.prepare('train',col_set=['feature','meta']);he=dataset.prepare('held',col_set=['feature','meta']);assert len(tr)==len(train) and len(he)==len(held)
state={'status':'started','plan_sha256':sha(card/'plan.json'),'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,uri=str(tracking)):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset,source_audit=audit);rec.save_objects(artifact_path='analysis',daily=daily,source_first_hour=detail)
 rec.log_metrics(training_known=int(train.source_known.sum()),held_known=int(held.source_known.sum()),model_fits=0,native_accounts=0)
 with R.uri_context(str(tracking)):reread=R.get_recorder(recorder_id=rec.id,experiment_id=str(rec.experiment_id)).load_object('analysis/daily')
 assert reread.equals(daily)
summary={'experiment':card.name,'decision':'conditional_peer_breadth_source_gate_passed_first_receipt_unqualified_no_fit','objective_achieved':False,'source_audit':'source_audit.json','source_calendar_days':len(days),'training_calendar_days':len(train),'training_known_days':int(train.source_known.sum()),'held_calendar_days':len(held),'held_known_days':int(held.source_known.sum()),'unknown_training_dates':audit['unknown_train_dates'],'unknown_held_dates':audit['unknown_held_dates'],'first_receipt_finality_qualified':False,'new_model_fits':0,'new_native_accounts':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/source_first_hour']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':card.name,'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='peer-breadth-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  names=[x.name for x in source.getmembers() if x.isfile()];assert any(x.endswith('/artifacts/analysis/daily') for x in names) and any(x.endswith('/artifacts/inputs/dataset') for x in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['NAS_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'decision':summary['decision'],'training_known':summary['training_known_days'],'held_known':summary['held_known_days'],'recorder_id':state['recorder_id'],'NAS_sha256':evidence['sha256']})


2026/10/23 04:34:48 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[82760:MainThread](2026-10-23 04:34:49,110) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[82760:MainThread](2026-10-23 04:34:49,112) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[82760:MainThread](2026-10-23 04:34:49,112) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/peer_5min')}


...

[82760:MainThread](2026-10-23 04:34:52,204) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_first_hour_peer_breadth_action_source_20261023.


[82760:MainThread](2026-10-23 04:34:52,206) INFO - qlib.workflow - [exp.py:258] - Experiment 750483563910065241 starts running ...


[82760:MainThread](2026-10-23 04:34:52,282) INFO - qlib.workflow - [recorder.py:345] - Recorder de73b6f2f7de4fe385ac1178674efefc starts running under Experiment 750483563910065241 ...


[82760:MainThread](2026-10-23 04:34:53,067) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'decision': 'conditional_peer_breadth_source_gate_passed_first_receipt_unqualified_no_fit', 'training_known': 1061, 'held_known': 944, 'recorder_id': 'de73b6f2f7de4fe385ac1178674efefc', 'NAS_sha256': 'bbadc66d6f50c96f6db2ed136d833b0042f0a8d750601ea4fdfe9fb62873145b'}


# Stage1 10:32 恢复动作：首小时半导体同业同涨来源

固定 AMD、NVDA、SMH 三个已列明同业/行业 ETF，取 09:35–10:30 各 12 根完整 5 分钟 bar 的首小时收益是否为正，同涨比例取三者均值。所有价格、质量标志和时间来自原生 Qlib 5min 来源；10:30 bar 完成后按研究口径 10:31 可得，早于 10:32 恢复决策。

来源覆盖通过：2018-10-15 至 2022-12-30 的 1,061 个恢复训练动作日已知 1,061 日，2023-01-04 至 2026-10-08 的 944 个复用历史动作日已知 944 日。不存在因应力日删除或填补的未知行；全源 2,006 日保留。0 标签读取、0 拟合、0 账户。

这一坐标衡量跨证券参与度，与已失败的 SOXX 本身带方向成交量不同；旧恢复 21 项含 SOXL/SOXX 早盘价格和无方向总量，但不含这三只同业同时上涨的比例。来源同涨并不证明净恢复价值。历史 5min bar 的首次接收和最终版本时钟仍未认证，不能用于账户晋升。

Qlib Recorder `de73b6f2f7de4fe385ac1178674efefc` 和 NAS 限定回读 SHA-256 `bbadc66d6f50c96f6db2ed136d833b0042f0a8d750601ea4fdfe9fb62873145b` 已验证。下一步在已冻结的 0bp Tiered、beta0.6 同状态净恢复/HOLD 标签上，保留旧 21 项输入与完整动作日历，单独冻结一版增量信息检验；通过后仍需原始 Stage1 完整 0bp 账户和合同 v2 全部门槛。